# Advanced GPT-5.4 Use Cases: Finance & Healthcare

Practical demonstrations of GPT-5.4's reasoning capabilities applied to finance and healthcare domains using the Azure OpenAI Responses API.

## 0 · Setup

In [1]:
import json, time, textwrap
from IPython.display import display, Markdown
from config import DEPLOYMENT, get_client

client = get_client()

def ask(prompt, *, effort="high", schema=None, system=None, prev_id=None):
    """Reusable helper — returns (response, elapsed)."""
    kwargs = {"model": DEPLOYMENT, "input": prompt, "reasoning": {"effort": effort}}
    if system:
        kwargs["instructions"] = system
    if schema:
        kwargs["text"] = {"format": {"type": "json_schema", "name": schema["name"],
                                      "schema": schema["schema"], "strict": True}}
    if prev_id:
        kwargs["previous_response_id"] = prev_id
    t0 = time.perf_counter()
    r = client.responses.create(**kwargs)
    return r, round(time.perf_counter() - t0, 1)

def show(resp, elapsed, raw_json=False):
    """Display response with metadata."""
    text = resp.output_text
    if raw_json:
        try:
            text = "```json\n" + json.dumps(json.loads(text), indent=2) + "\n```"
        except (json.JSONDecodeError, TypeError):
            text = f"⚠️ Expected JSON but got:\n\n{text or '(empty response)'}"
    display(Markdown(text))
    u = resp.usage
    print(f"⏱ {elapsed}s | in: {u.input_tokens}  out: {u.output_tokens}  total: {u.total_tokens}")

---
# FINANCE USE CASES

## 1 · Financial Report Summarization & Key Metrics Extraction

Extract structured metrics from a 10-K excerpt using JSON schema output.

In [2]:
filing_excerpt = """
CONTOSO CORP — Form 10-K (FY 2025)

Revenue for fiscal year 2025 was $48.2 billion, up 12% from $43.0 billion in FY 2024.
Cost of revenue increased to $19.3 billion (FY 2024: $17.8 billion). Gross margin improved
to 59.9% from 58.6%. Operating expenses totaled $14.1 billion including $5.2B in R&D
(up 18%) and $4.8B in SG&A. Operating income reached $14.8 billion (30.7% margin).
Interest expense was $0.9B; other income $0.3B. Pre-tax income: $14.2B.
Income tax provision: $2.8B (effective rate 19.7%). Net income: $11.4 billion,
up from $9.6 billion in FY 2024. Diluted EPS: $15.22 (FY 2024: $12.80).
EBITDA: $17.4 billion. Free cash flow: $13.1 billion. Total debt: $22.5 billion.
Cash and equivalents: $18.7 billion. The company repurchased $6.2B in shares and
paid $3.8B in dividends. Management guided FY 2026 revenue of $53-55 billion.
"""

financial_schema = {
    "name": "financial_metrics",
    "schema": {
        "type": "object",
        "properties": {
            "company": {"type": "string"},
            "fiscal_year": {"type": "string"},
            "revenue_B": {"type": "number"},
            "revenue_yoy_pct": {"type": "number"},
            "gross_margin_pct": {"type": "number"},
            "operating_income_B": {"type": "number"},
            "net_income_B": {"type": "number"},
            "net_income_yoy_pct": {"type": "number"},
            "ebitda_B": {"type": "number"},
            "diluted_eps": {"type": "number"},
            "free_cash_flow_B": {"type": "number"},
            "total_debt_B": {"type": "number"},
            "cash_B": {"type": "number"},
            "forward_guidance": {"type": "string"}
        },
        "required": ["company","fiscal_year","revenue_B","revenue_yoy_pct",
                      "gross_margin_pct","operating_income_B","net_income_B",
                      "net_income_yoy_pct","ebitda_B","diluted_eps",
                      "free_cash_flow_B","total_debt_B","cash_B","forward_guidance"],
        "additionalProperties": False
    }
}

r, t = ask(f"Extract key financial metrics from this 10-K excerpt:\n\n{filing_excerpt}",
           schema=financial_schema)
show(r, t, raw_json=True)

```json
{
  "company": "CONTOSO CORP",
  "fiscal_year": "2025",
  "revenue_B": 48.2,
  "revenue_yoy_pct": 12,
  "gross_margin_pct": 59.9,
  "operating_income_B": 14.8,
  "net_income_B": 11.4,
  "net_income_yoy_pct": 18.8,
  "ebitda_B": 17.4,
  "diluted_eps": 15.22,
  "free_cash_flow_B": 13.1,
  "total_debt_B": 22.5,
  "cash_B": 18.7,
  "forward_guidance": "FY 2026 revenue of $53-55 billion"
}
```

⏱ 8.5s | in: 455  out: 322  total: 777


## 2 · Automated Credit Risk Assessment

Assess credit risk from unstructured loan narratives and news signals.

In [3]:
credit_data = """
LOAN APPLICATION — Borrower: Acme Manufacturing LLC
Requested: $4.5M term loan (5-year) for equipment expansion.
Annual revenue: $12M (down from $14M last year). Current debt: $2.1M revolving line.
Borrower narrative: \"We lost our second-largest customer (18% of revenue) in Q3 but
have signed two new contracts starting Q2 next year worth ~$5M annually combined.
We need the equipment to fulfill these contracts.\"

RECENT NEWS:
- \"Acme Manufacturing faces EPA investigation over waste disposal at Ohio plant\" (2 weeks ago)
- \"Regional manufacturing sector shows 8% growth in new orders\" (1 week ago)
- \"Acme Manufacturing CEO cited in local business journal for community investment\" (3 days ago)
"""

credit_schema = {
    "name": "credit_risk_assessment",
    "schema": {
        "type": "object",
        "properties": {
            "borrower": {"type": "string"},
            "risk_score": {"type": "integer"},
            "risk_rating": {"type": "string"},
            "recommendation": {"type": "string"},
            "positive_factors": {"type": "array", "items": {"type": "string"}},
            "risk_factors": {"type": "array", "items": {"type": "string"}},
            "mitigants": {"type": "array", "items": {"type": "string"}},
            "conditions": {"type": "array", "items": {"type": "string"}},
            "rationale": {"type": "string"}
        },
        "required": ["borrower","risk_score","risk_rating","recommendation",
                      "positive_factors","risk_factors","mitigants","conditions","rationale"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"You are a senior credit analyst. Assess the credit risk:\n\n{credit_data}",
    schema=credit_schema, effort="low"
)
show(r, t, raw_json=True)

```json
{
  "borrower": "Acme Manufacturing LLC",
  "risk_score": 62,
  "risk_rating": "Moderate-High",
  "recommendation": "Conditional approval with heightened diligence and tighter structure.",
  "positive_factors": [
    "Signed new contracts expected to generate approximately $5M annually, supporting future revenue replacement and growth",
    "Loan purpose is tied to equipment expansion needed to fulfill identified business opportunities",
    "Industry backdrop appears supportive, with regional manufacturing new orders up 8%",
    "Existing scale is meaningful, with current annual revenue of $12M"
  ],
  "risk_factors": [
    "Revenue declined from $14M to $12M year over year, indicating recent business weakness",
    "Loss of second-largest customer representing 18% of revenue increases concentration and execution risk",
    "New contracts do not begin until Q2 next year, creating timing and ramp-up risk",
    "Current leverage already includes $2.1M on a revolving line; adding $4.5M term debt could materially increase debt burden",
    "EPA investigation over waste disposal at the Ohio plant creates legal, regulatory, operational, and reputational risk",
    "Repayment capacity depends significantly on successful onboarding and performance of new contracts"
  ],
  "mitigants": [
    "Structure loan with covenant protections tied to leverage, fixed charge coverage, and minimum liquidity",
    "Require verification of the two new contracts, including term, termination rights, pricing, and customer credit quality",
    "Consider staged funding aligned with equipment delivery and contract commencement",
    "Obtain first-priority lien on financed equipment and broader collateral support where available",
    "Require environmental review, reserve, or indemnity to address EPA investigation exposure",
    "Use excess cash flow sweep to accelerate deleveraging if performance exceeds plan"
  ],
  "conditions": [
    "Provide executed copies of the two new customer contracts and evidence of purchase commitments",
    "Complete satisfactory environmental and legal due diligence regarding the EPA investigation",
    "Demonstrate pro forma debt service coverage at acceptable levels under base and downside cases",
    "Maintain the revolving line within agreed borrowing base and liquidity thresholds",
    "No material adverse change in customer awards, regulatory matters, or operating performance prior to closing",
    "Lender to receive regular reporting on contract ramp, capex deployment, and compliance status"
  ],
  "rationale": "Acme presents a mixed credit profile. The company has meaningful revenue scale and a clear use of proceeds tied to growth, and the two new contracts could more than offset the lost customer if they perform as expected. The sector backdrop is also constructive. However, credit risk is elevated by the recent revenue decline, customer concentration issues evidenced by the 18% customer loss, and the timing gap before new contracts begin contributing. The proposed $4.5M term loan would significantly increase leverage on top of the existing $2.1M revolver. Most importantly, the recent EPA investigation introduces uncertain contingent liabilities and potential operational disruption. On balance, the credit is supportable only with strong diligence, collateral, covenants, and confirmation that the new business is firm and sufficient to service the higher debt load."
}
```

⏱ 19.0s | in: 300  out: 632  total: 932


## 3 · Fraud Detection Narrative Analysis

Classify fraud likelihood from transaction descriptions and dispute text.

In [4]:
fraud_cases = """
CASE 1:
Transaction: $3,200 wire transfer to \"GlobalTech Services Ltd\" (first-time payee, registered in Belize).
Customer dispute: \"I never authorized this. I received a call from someone claiming to be my bank
asking me to verify my account by entering my PIN on the phone. The next day this charge appeared.\"

CASE 2:
Transaction: $89.99 recurring charge from \"StreamPlus Premium.\"
Customer dispute: \"I cancelled this subscription 3 months ago. I have the cancellation confirmation
email. They keep charging me every month.\"

CASE 3:
Transaction: 14 purchases totaling $12,400 at electronics retailers across 3 states within 6 hours.
Customer dispute: \"My card was in my wallet the entire time. I was at work all day.
I noticed the charges when I got an alert on my phone.\"
"""

fraud_schema = {
    "name": "fraud_analysis",
    "schema": {
        "type": "object",
        "properties": {
            "cases": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "case_id": {"type": "string"},
                        "fraud_likelihood": {"type": "string"},
                        "fraud_type": {"type": "string"},
                        "suspicious_indicators": {"type": "array", "items": {"type": "string"}},
                        "recommended_action": {"type": "string"},
                        "analyst_summary": {"type": "string"}
                    },
                    "required": ["case_id","fraud_likelihood","fraud_type",
                                 "suspicious_indicators","recommended_action","analyst_summary"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["cases"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"You are a fraud analyst. Analyze each case for fraud indicators:\n\n{fraud_cases}",
    schema=fraud_schema, effort="medium"
)
show(r, t, raw_json=True)

```json
{
  "cases": [
    {
      "case_id": "CASE 1",
      "fraud_likelihood": "High",
      "fraud_type": "Social engineering / account takeover leading to unauthorized wire fraud",
      "suspicious_indicators": [
        "Customer reports an unsolicited call from someone impersonating the bank",
        "Customer was asked to disclose/enter a PIN, which banks typically do not request by phone",
        "High-value wire transfer",
        "First-time payee",
        "Payee is an offshore entity registered in Belize",
        "Customer states the transaction was not authorized",
        "Transaction occurred shortly after credential/PIN compromise event"
      ],
      "recommended_action": "Treat as likely fraud. Immediately freeze or secure the affected account, block further outgoing transfers, reset credentials/PIN, review for additional unauthorized activity, and attempt wire recall or beneficiary bank contact. Escalate to the fraud investigations team and document as impersonation/social engineering fraud.",
      "analyst_summary": "This case shows strong indicators of bank-impersonation scam activity. The customer was deceived into providing sensitive authentication information, and a high-dollar wire was then sent to a new offshore beneficiary. The timing, method, and destination make this highly likely to be unauthorized fraud rather than a legitimate customer-initiated payment."
    },
    {
      "case_id": "CASE 2",
      "fraud_likelihood": "Medium",
      "fraud_type": "Merchant billing dispute / recurring charge after cancellation",
      "suspicious_indicators": [
        "Recurring charge continues after the customer states the subscription was cancelled",
        "Customer says they possess a cancellation confirmation email",
        "Same merchant appears to be billing monthly despite cancellation",
        "Charge amount is consistent with a subscription rather than account takeover behavior"
      ],
      "recommended_action": "Classify initially as a merchant dispute rather than clear third-party fraud. Request and review the cancellation confirmation, cancellation date, and terms of service. If documentation confirms cancellation before the disputed billing dates, pursue chargeback/dispute for cancelled recurring transactions and consider blocking future charges from the merchant.",
      "analyst_summary": "This appears more consistent with a continued billing or merchant compliance issue than classic payment fraud. The key evidence is the customer\u2019s cancellation confirmation. If verified, the charges are likely improper recurring billings after cancellation, but not necessarily the result of stolen credentials or account takeover."
    },
    {
      "case_id": "CASE 3",
      "fraud_likelihood": "High",
      "fraud_type": "Counterfeit card / card-present fraud or compromised card credentials",
      "suspicious_indicators": [
        "14 separate purchases in a very short period",
        "Total loss amount is high at $12,400",
        "Transactions occurred across 3 states within 6 hours",
        "Customer states the physical card remained in their possession",
        "Purchase pattern targets electronics retailers, a common fraud-resale category",
        "Customer was at work and received real-time alerts, indicating likely unauthorized use"
      ],
      "recommended_action": "Treat as highly likely card fraud. Immediately block the card, issue a replacement, identify whether transactions were card-present or card-not-present, review authorization data for chip/swipe/manual entry patterns, and dispute/charge back unauthorized purchases. Check for prior compromise points and monitor related accounts for further fraud.",
      "analyst_summary": "The geographic impossibility, rapid velocity, and concentration at electronics merchants strongly indicate fraudulent use. Because the customer still had the card, this suggests counterfeit card usage or compromised card data rather than simple lost/stolen card misuse. This should be escalated as a likely confirmed unauthorized card fraud case."
    }
  ]
}
```

⏱ 22.2s | in: 287  out: 708  total: 995


## 4 · Portfolio Sentiment from Earnings Calls

Batch-analyze earnings call snippets for sentiment and forward guidance signals.

In [5]:
earnings_snippets = """
COMPANY A (Cloud Infrastructure) — CEO remarks:
\"We saw 28% growth in our cloud segment, exceeding guidance. Enterprise deal sizes increased
40% QoQ. However, we're seeing some elongation in sales cycles for mid-market. We expect
next quarter revenue of $8.2-8.5B and are raising full-year guidance by $1.2B.\"

COMPANY B (Pharmaceutical) — CFO remarks:
\"Oncology revenue grew 6% but missed consensus by $200M due to competitive pressure from
biosimilars. Our pipeline has 3 Phase III readouts in H2. We're maintaining guidance but
flagging FX headwinds of approximately $400M for the full year. Gross margins compressed
150bps due to manufacturing cost inflation.\"

COMPANY C (Regional Bank) — CEO remarks:
\"Net interest margin expanded 22bps to 3.41%. Loan growth was 11% annualized, primarily in
commercial real estate. Credit quality remains strong — NPLs at 0.3%. However, we increased
our provision for credit losses by $45M given macroeconomic uncertainty. We expect NIM
stability in Q4 but are cautious on 2026 given the rate environment.\"
"""

sentiment_schema = {
    "name": "earnings_sentiment",
    "schema": {
        "type": "object",
        "properties": {
            "companies": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "company": {"type": "string"},
                        "sector": {"type": "string"},
                        "overall_sentiment": {"type": "string"},
                        "sentiment_score": {"type": "number"},
                        "positive_signals": {"type": "array", "items": {"type": "string"}},
                        "negative_signals": {"type": "array", "items": {"type": "string"}},
                        "guidance_direction": {"type": "string"},
                        "key_risk": {"type": "string"}
                    },
                    "required": ["company","sector","overall_sentiment","sentiment_score",
                                 "positive_signals","negative_signals","guidance_direction","key_risk"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["companies"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"Analyze sentiment and forward guidance from these earnings call excerpts:\n\n{earnings_snippets}",
    schema=sentiment_schema, effort="medium"
)
show(r, t, raw_json=True)

```json
{
  "companies": [
    {
      "company": "COMPANY A",
      "sector": "Cloud Infrastructure",
      "overall_sentiment": "Positive",
      "sentiment_score": 0.78,
      "positive_signals": [
        "Cloud segment growth of 28%, exceeding guidance",
        "Enterprise deal sizes increased 40% quarter over quarter",
        "Next-quarter revenue outlook of $8.2-8.5B implies continued momentum",
        "Full-year guidance raised by $1.2B"
      ],
      "negative_signals": [
        "Sales cycles are elongating in the mid-market segment"
      ],
      "guidance_direction": "Raised",
      "key_risk": "Softening mid-market demand and longer sales cycles could pressure conversion and near-term bookings."
    },
    {
      "company": "COMPANY B",
      "sector": "Pharmaceutical",
      "overall_sentiment": "Mixed to Negative",
      "sentiment_score": -0.22,
      "positive_signals": [
        "Oncology revenue still grew 6%",
        "Pipeline includes 3 Phase III readouts in H2, offering potential catalysts"
      ],
      "negative_signals": [
        "Oncology revenue missed consensus by $200M",
        "Competitive pressure from biosimilars is weighing on sales",
        "Guidance was only maintained rather than raised",
        "FX headwinds of approximately $400M for the full year",
        "Gross margins compressed 150bps due to manufacturing cost inflation"
      ],
      "guidance_direction": "Maintained",
      "key_risk": "Biosimilar competition combined with FX pressure and margin compression could continue to weigh on earnings."
    },
    {
      "company": "COMPANY C",
      "sector": "Regional Bank",
      "overall_sentiment": "Mixed",
      "sentiment_score": 0.18,
      "positive_signals": [
        "Net interest margin expanded 22bps to 3.41%",
        "Loan growth was 11% annualized",
        "Credit quality remains strong with NPLs at 0.3%",
        "Management expects NIM stability in Q4"
      ],
      "negative_signals": [
        "Provision for credit losses increased by $45M",
        "Commercial real estate is the primary driver of loan growth, which adds sector concentration risk",
        "Management is cautious on 2026 given the rate environment and macro uncertainty"
      ],
      "guidance_direction": "Stable near term, cautious longer term",
      "key_risk": "Higher credit costs and exposure to commercial real estate amid macro and rate uncertainty."
    }
  ]
}
```

⏱ 11.7s | in: 383  out: 608  total: 991


---
# HEALTHCARE USE CASES

## 5 · Clinical Note Structuring & ICD Code Suggestion

Parse free-text clinical notes into structured fields with ICD-10 code suggestions.

In [6]:
clinical_note = """
Patient: 67 y/o female presents to ED with acute onset chest pain radiating to left arm,
started 2 hours ago while gardening. Associated diaphoresis and nausea. Denies SOB.
PMH: HTN x 15 years, T2DM on metformin, hyperlipidemia on atorvastatin, former smoker
(quit 5 years ago, 30 pack-year history). Family hx: father MI at age 58.
Vitals: BP 168/94, HR 96, RR 18, SpO2 97% on RA, Temp 98.4F.
ECG: ST elevation in leads II, III, aVF. Troponin I: 2.4 ng/mL (normal <0.04).
Assessment: STEMI — inferior wall. Cardiology consulted, patient taken for emergent PCI.
Heparin drip and dual antiplatelet therapy initiated. Admit to CCU post-procedure.
"""

clinical_schema = {
    "name": "clinical_note_structured",
    "schema": {
        "type": "object",
        "properties": {
            "demographics": {"type": "object", "properties": {
                "age": {"type": "integer"}, "sex": {"type": "string"}, "setting": {"type": "string"}
            }, "required": ["age","sex","setting"], "additionalProperties": False},
            "chief_complaint": {"type": "string"},
            "hpi": {"type": "string"},
            "past_medical_history": {"type": "array", "items": {"type": "string"}},
            "vitals": {"type": "object", "properties": {
                "bp": {"type": "string"}, "hr": {"type": "integer"}, "rr": {"type": "integer"},
                "spo2": {"type": "string"}, "temp": {"type": "string"}
            }, "required": ["bp","hr","rr","spo2","temp"], "additionalProperties": False},
            "key_findings": {"type": "array", "items": {"type": "string"}},
            "assessment": {"type": "string"},
            "plan": {"type": "array", "items": {"type": "string"}},
            "icd10_suggestions": {"type": "array", "items": {
                "type": "object", "properties": {
                    "code": {"type": "string"},
                    "description": {"type": "string"},
                    "confidence": {"type": "string"}
                }, "required": ["code","description","confidence"], "additionalProperties": False
            }}
        },
        "required": ["demographics","chief_complaint","hpi","past_medical_history",
                      "vitals","key_findings","assessment","plan","icd10_suggestions"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"Structure this clinical note and suggest ICD-10 codes:\n\n{clinical_note}",
    schema=clinical_schema, effort="medium",
    system="You are a clinical documentation specialist. Extract structured data and suggest ICD-10 codes with confidence (high/medium/low)."
)
show(r, t, raw_json=True)

```json
{
  "demographics": {
    "age": 67,
    "sex": "female",
    "setting": "Emergency Department"
  },
  "chief_complaint": "Acute chest pain radiating to the left arm",
  "hpi": "67-year-old female presents to the ED with acute onset chest pain radiating to the left arm, beginning 2 hours prior while gardening. Associated symptoms include diaphoresis and nausea. She denies shortness of breath. Significant cardiovascular risk factors include hypertension, type 2 diabetes mellitus, hyperlipidemia, former smoking history, and family history of myocardial infarction in her father at age 58.",
  "past_medical_history": [
    "Hypertension x15 years",
    "Type 2 diabetes mellitus on metformin",
    "Hyperlipidemia on atorvastatin",
    "Former smoker; quit 5 years ago; 30 pack-year history",
    "Family history of myocardial infarction (father at age 58)"
  ],
  "vitals": {
    "bp": "168/94",
    "hr": 96,
    "rr": 18,
    "spo2": "97% on room air",
    "temp": "98.4 F"
  },
  "key_findings": [
    "ECG with ST elevation in leads II, III, and aVF",
    "Troponin I 2.4 ng/mL (normal <0.04)",
    "Clinical presentation consistent with acute inferior wall STEMI",
    "No shortness of breath reported"
  ],
  "assessment": "Acute inferior wall ST-elevation myocardial infarction (STEMI) with elevated troponin, in a patient with multiple cardiovascular risk factors including hypertension, type 2 diabetes mellitus, hyperlipidemia, prior tobacco use, and family history of premature myocardial infarction.",
  "plan": [
    "Cardiology consulted",
    "Taken for emergent percutaneous coronary intervention (PCI)",
    "Heparin drip initiated",
    "Dual antiplatelet therapy initiated",
    "Admit to CCU post-procedure",
    "Ongoing monitoring and post-PCI management"
  ],
  "icd10_suggestions": [
    {
      "code": "I21.19",
      "description": "ST elevation (STEMI) myocardial infarction involving other coronary artery of inferior wall",
      "confidence": "medium"
    },
    {
      "code": "I10",
      "description": "Essential (primary) hypertension",
      "confidence": "high"
    },
    {
      "code": "E11.9",
      "description": "Type 2 diabetes mellitus without complications",
      "confidence": "high"
    },
    {
      "code": "E78.5",
      "description": "Hyperlipidemia, unspecified",
      "confidence": "high"
    },
    {
      "code": "Z87.891",
      "description": "Personal history of nicotine dependence",
      "confidence": "high"
    },
    {
      "code": "Z82.49",
      "description": "Family history of ischemic heart disease and other diseases of the circulatory system",
      "confidence": "medium"
    },
    {
      "code": "Z79.84",
      "description": "Long term (current) use of oral hypoglycemic drugs",
      "confidence": "medium"
    }
  ]
}
```

⏱ 37.9s | in: 444  out: 1708  total: 2152


## 6 · Medical Literature Review & Evidence Synthesis

Synthesize findings from multiple study abstracts, highlight conflicts, and rate evidence strength.

In [7]:
abstracts = """
STUDY 1 (RCT, n=2,400, NEJM 2025):
GLP-1 receptor agonist semaglutide 2.4mg weekly vs placebo in patients with HFpEF and obesity.
Primary endpoint (composite of CV death or HF hospitalization) reduced by 22% (HR 0.78, 95% CI
0.65-0.93, p=0.006). Significant weight loss (-13.2% vs -2.1%). GI adverse events in 38% vs 12%.

STUDY 2 (Observational cohort, n=48,000, Lancet 2025):
Real-world analysis of GLP-1 RA use in HFpEF patients across 12 health systems. Associated with
18% reduction in all-cause mortality (adjusted HR 0.82, 0.76-0.89). However, higher rates of
acute kidney injury (OR 1.31, 1.12-1.53) and pancreatitis (OR 1.89, 1.22-2.93) observed.

STUDY 3 (Meta-analysis of 6 RCTs, n=8,200, JAMA 2025):
Pooled analysis showed 19% reduction in HF hospitalization (RR 0.81, 0.72-0.91) but no significant
effect on CV mortality (RR 0.94, 0.83-1.07). Subgroup analysis showed benefit concentrated in
patients with BMI >35. Discontinuation rate due to GI side effects: 15%.
"""

r, t = ask(
    f"Synthesize the evidence from these studies on GLP-1 agonists in HFpEF. "
    f"Highlight agreements, conflicts, and rate overall evidence strength.\n\n{abstracts}",
    effort="medium",
    system="You are a clinical evidence reviewer. Be precise about statistical findings. Use markdown formatting."
)
show(r, t)

## Bottom line

The evidence **supports GLP-1 receptor agonists, especially semaglutide, as beneficial for HFpEF patients with obesity**, with the **strongest and most consistent signal being fewer HF hospitalizations / better heart-failure clinical outcomes**, not mortality reduction.  

The main tradeoff is **frequent GI toxicity**, while **possible rare serious harms** such as AKI and pancreatitis are suggested by observational data but are less firmly established in randomized trials.

---

## Evidence summary by study

| Study | Design | Population | Main efficacy findings | Main safety findings | Key interpretation |
|---|---|---|---|---|---|
| **Study 1** | RCT, n=2,400 | HFpEF + obesity | Primary composite of **CV death or HF hospitalization** reduced: **HR 0.78 (95% CI 0.65–0.93, p=0.006)** = 22% relative reduction | **GI adverse events 38% vs 12%**; weight loss **-13.2% vs -2.1%** | Best direct causal evidence that semaglutide improves major HF-related outcomes in obese HFpEF |
| **Study 2** | Observational cohort, n=48,000 | Real-world HFpEF across 12 systems | **All-cause mortality** lower with GLP-1 RA use: **adjusted HR 0.82 (95% CI 0.76–0.89)** | Higher **AKI**: **OR 1.31 (1.12–1.53)**; higher **pancreatitis**: **OR 1.89 (1.22–2.93)** | Suggests effectiveness in practice, but confounding limits causal inference; raises safety concerns for renal/pancreatic events |
| **Study 3** | Meta-analysis of 6 RCTs, n=8,200 | HFpEF trials | **HF hospitalization** reduced: **RR 0.81 (0.72–0.91)**; **CV mortality not significant**: **RR 0.94 (0.83–1.07)** | **GI-related discontinuation 15%** | Reinforces hospitalization benefit; does **not** support a clear CV mortality benefit; benefit stronger when **BMI >35** |

---

## Where the studies agree

### 1. **Benefit is most consistent for HF-related morbidity**
There is strong directional consistency that GLP-1 agonists improve HFpEF clinical outcomes, especially **HF hospitalization**:

- **RCT (Study 1):** composite CV death/HF hospitalization improved, **HR 0.78**
- **Meta-analysis of RCTs (Study 3):** HF hospitalization reduced, **RR 0.81**

These findings align well and suggest the effect on the Study 1 composite is likely driven at least in part by **reduced HF hospitalization**.

### 2. **Obesity appears to be an important treatment-response modifier**
The most convincing randomized evidence is in **HFpEF with obesity**:

- Study 1 enrolled **HFpEF patients with obesity**
- Study 3 found benefit concentrated in **BMI >35**

This supports the idea that GLP-1 agonists may be particularly effective in the **obese HFpEF phenotype**, which is biologically plausible given the large weight-loss effect in Study 1 (**-13.2% vs -2.1%**).

### 3. **GI adverse effects are common**
Safety signals are consistent for gastrointestinal intolerance:

- Study 1: **GI adverse events 38% vs 12%**
- Study 3: **15% discontinuation** due to GI side effects

So GI toxicity is a **robust and reproducible adverse effect**, and likely clinically important for adherence.

---

## Where the studies conflict or diverge

### 1. **Mortality benefit is uncertain**
This is the main area of tension.

- **Study 2** found lower **all-cause mortality** with GLP-1 RA use: **adjusted HR 0.82 (0.76–0.89)**
- But **Study 3** found **no significant reduction in CV mortality**: **RR 0.94 (0.83–1.07)**

### How to interpret this conflict
This is **not necessarily a direct contradiction**, because:

- The endpoints differ:
  - **All-cause mortality** (Study 2)
  - **CV mortality** (Study 3)
- Study 2 is **observational**, so residual confounding is possible:
  - GLP-1 users may differ in obesity treatment intensity, diabetes care, access to care, or overall cardiometabolic risk management
- Study 1’s primary endpoint was **composite**, so it does not prove the mortality component alone improved

**Net interpretation:** mortality benefit remains **possible but unproven**, and current randomized evidence does **not** establish a clear reduction in CV death.

### 2. **Serious safety signals beyond GI effects are less settled**
- Study 2 reported:
  - **AKI:** **OR 1.31 (1.12–1.53)**
  - **Pancreatitis:** **OR 1.89 (1.22–2.93)**
- These events were **not highlighted** in the RCT/meta-analysis summaries

Possible explanations:
- RCTs may have been underpowered for relatively rare harms
- Real-world populations may include higher-risk patients
- Observational designs can also over- or underestimate harms because of differential surveillance and residual confounding

**Net interpretation:** the renal/pancreatic safety signals are **concerning but less definitive** than the GI toxicity signal.

---

## Integrated interpretation

### Efficacy
The highest-quality evidence supports GLP-1 agonists as improving **HFpEF outcomes mainly by reducing HF hospitalization / worsening HF events**, especially in patients with **obesity or very high BMI**.

The RCT and meta-analysis are broadly concordant:

- **Study 1:** composite CV death/HF hospitalization improved
- **Study 3:** HF hospitalization improved, but **CV mortality not significantly changed**

This pattern suggests the clinical benefit is likely driven more by **symptom/congestion-related or hospitalization-related improvements** than by a proven mortality effect.

### Safety
- **Common harm:** GI adverse effects are clearly increased
- **Potential less-common harms:** AKI and pancreatitis may be increased, but evidence is less secure because it comes mainly from observational data

### Applicability
The evidence is strongest for:
- **HFpEF with obesity**
- likely especially **BMI >35**

Generalization to:
- leaner HFpEF populations
- non-obese phenotypes
- long-term mortality prevention

is more uncertain.

---

## Overall evidence strength

### 1. **For reducing HF hospitalization / improving HF clinical outcomes: Moderate to high**
**Why:**
- Supported by a large **RCT**
- Reinforced by a **meta-analysis of 6 RCTs**
- Effect sizes are reasonably consistent:
  - **HR 0.78** for composite in Study 1
  - **RR 0.81** for HF hospitalization in Study 3

### 2. **For reducing mortality: Low to moderate / inconclusive**
**Why:**
- Observational data suggest benefit:
  - **all-cause mortality HR 0.82**
- But randomized evidence does **not** show significant **CV mortality** reduction:
  - **RR 0.94 (0.83–1.07)**

So mortality benefit is **hypothesis-supporting**, not established.

### 3. **For GI adverse effects: High**
**Why:**
- Consistent across randomized evidence
- Large absolute difference in Study 1:
  - **38% vs 12%**
- Meta-analysis shows meaningful discontinuation:
  - **15%**

### 4. **For AKI and pancreatitis risk: Low to moderate**
**Why:**
- Signal present in a very large observational cohort
- But not yet firmly corroborated by randomized data in the summary provided

---

## Practical conclusion

**Overall, the evidence favors GLP-1 agonists in obese HFpEF for reducing HF-related events, with the strongest support for lowering HF hospitalization rather than mortality.** The benefit appears most credible in patients with **marked obesity**, which is consistent across the RCT and the meta-analysis.  

However, treatment comes with a **clear GI tolerability burden**, and clinicians should remain alert to **possible AKI and pancreatitis**, which are suggested in real-world data but not yet definitively established in trials.

## One-sentence judgment
**Overall evidence strength: moderate for efficacy in obese HFpEF, weak-to-inconclusive for mortality reduction, high for GI harms, and low-to-moderate for rare serious harms.**

If you want, I can also convert this into a **GRADE-style evidence table** or a **journal-club style critique**.

⏱ 53.6s | in: 396  out: 2251  total: 2647


## 7 · Patient Discharge Summary Generation

Generate a discharge summary from structured EHR data.

In [8]:
ehr_data = """
PATIENT: James M., 72M | MRN: 4829173 | Admitted: 2026-02-28 | Discharged: 2026-03-05
ADMITTING DX: Community-acquired pneumonia (CAP)

LABS (admission → discharge):
- WBC: 18.2 → 8.4 K/uL | CRP: 142 → 18 mg/L | Procalcitonin: 2.8 → 0.3 ng/mL
- Creatinine: 1.4 → 1.1 mg/dL | BUN: 32 → 18 mg/dL
- Blood culture: Streptococcus pneumoniae (sensitive to ceftriaxone, levofloxacin)

VITALS (discharge): BP 128/76, HR 78, RR 16, SpO2 96% on RA, Temp 98.6F

IMAGING: CXR admission — RLL consolidation with small parapneumonic effusion.
CXR day 5 — improving consolidation, effusion resolved.

TREATMENT:
- IV ceftriaxone 2g daily x 5 days → transitioned to PO amoxicillin-clavulanate
- IV fluids (LR) x 48h then discontinued
- Incentive spirometry, early ambulation
- Home O2 not required at discharge

COMORBIDITIES: COPD (on tiotropium), T2DM (A1c 7.2%), BPH
HOME MEDS: Tiotropium, metformin 1000mg BID, tamsulosin 0.4mg daily, aspirin 81mg
"""

r, t = ask(
    f"Generate a concise discharge summary with follow-up instructions:\n\n{ehr_data}",
    effort="medium",
    system="You are a hospitalist. Write a professional discharge summary in standard clinical format. Include: hospital course, discharge medications, follow-up plan, return precautions."
)
show(r, t)

**Discharge Summary**

**Patient:** James M.  
**MRN:** 4829173  
**Age/Sex:** 72M  
**Admit Date:** 2026-02-28  
**Discharge Date:** 2026-03-05  

**Discharge Diagnosis:**  
1. Community-acquired pneumonia, right lower lobe, due to *Streptococcus pneumoniae*  
2. Small parapneumonic effusion, resolved  
3. Mild acute kidney injury/dehydration, resolved  
4. COPD  
5. Type 2 diabetes mellitus  
6. BPH  

**Hospital Course:**  
Patient was admitted with community-acquired pneumonia. Admission CXR showed right lower lobe consolidation with a small parapneumonic effusion. Blood cultures grew *Streptococcus pneumoniae* sensitive to ceftriaxone and levofloxacin. He was treated with IV ceftriaxone 2 g daily for 5 days, IV lactated Ringer’s for 48 hours, incentive spirometry, and early ambulation. He improved clinically with normalization of inflammatory markers and leukocytosis (WBC 18.2 → 8.4, CRP 142 → 18, procalcitonin 2.8 → 0.3). Renal function improved as well (Cr 1.4 → 1.1, BUN 32 → 18). Repeat CXR on hospital day 5 showed improving consolidation and resolution of the effusion. He remained hemodynamically stable and was discharged on room air; home oxygen was not required.

**Condition at Discharge:**  
Improved and clinically stable.  
Discharge vitals: BP 128/76, HR 78, RR 16, SpO2 96% on room air, Temp 98.6 F.

**Discharge Medications:**  
- **Amoxicillin-clavulanate**: continue oral course as prescribed to complete treatment for pneumonia  
- **Tiotropium**: continue home dose  
- **Metformin 1000 mg PO BID**  
- **Tamsulosin 0.4 mg PO daily**  
- **Aspirin 81 mg PO daily**

**Follow-Up Plan:**  
- Follow up with primary care physician within **1 week** after discharge  
- Review final culture results/symptom resolution and ensure completion of antibiotic course  
- Repeat chest imaging only if symptoms fail to resolve or worsen  
- Continue COPD maintenance inhaler and usual diabetes medications

**Discharge Instructions / Return Precautions:**  
Return to the ED or seek urgent care for:  
- Worsening shortness of breath or new oxygen requirement  
- Fever, chills, or recurrent productive cough  
- Chest pain  
- Confusion, dizziness, or fainting  
- Inability to keep down medications/fluids  
- Any other worsening symptoms

**Disposition:** Home

⏱ 25.2s | in: 399  out: 998  total: 1397


## 8 · Adverse Drug Reaction Detection from Patient Feedback

Identify ADRs from patient forum posts, classify severity, and map to MedDRA terms.

In [9]:
patient_posts = """
POST 1: \"Been on Lisinopril 20mg for 3 months. Developed this dry persistent cough that
won't go away. My doctor says it might be the medication. Also noticed some dizziness
when standing up quickly.\"

POST 2: \"Started Methotrexate for my RA 6 weeks ago. Having severe nausea after each
dose, mouth sores, and my latest blood work showed my liver enzymes are 3x the upper
limit. My rheumatologist is \"monitoring closely.\" Feeling really fatigued.\"

POST 3: \"Taking Sertraline 100mg for anxiety. First 2 weeks were rough — couldn't sleep,
felt jittery, and had weird vivid dreams. Now 8 weeks in and feeling much better but
gained 12 pounds and my libido is completely gone.\"

POST 4: \"My mom (78) was prescribed Warfarin after her valve replacement. She fell last
week and got this massive bruise on her hip that hasn't gone away. She also mentioned
seeing blood in her urine for the past few days. We're going to the ER tonight.\"
"""

adr_schema = {
    "name": "adr_detection",
    "schema": {
        "type": "object",
        "properties": {
            "reports": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "post_id": {"type": "string"},
                        "drug": {"type": "string"},
                        "adverse_reactions": {"type": "array", "items": {
                            "type": "object", "properties": {
                                "reaction": {"type": "string"},
                                "meddra_term": {"type": "string"},
                                "severity": {"type": "string"},
                                "known_adr": {"type": "boolean"}
                            }, "required": ["reaction","meddra_term","severity","known_adr"],
                               "additionalProperties": False
                        }},
                        "urgency": {"type": "string"},
                        "recommended_action": {"type": "string"}
                    },
                    "required": ["post_id","drug","adverse_reactions","urgency","recommended_action"],
                    "additionalProperties": False
                }
            }
        },
        "required": ["reports"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"Analyze these patient posts for adverse drug reactions:\n\n{patient_posts}",
    schema=adr_schema, effort="medium",
    system="You are a pharmacovigilance specialist. Classify severity as mild/moderate/severe/life-threatening. Map reactions to MedDRA preferred terms."
)
show(r, t, raw_json=True)

```json
{
  "reports": [
    {
      "post_id": "1",
      "drug": "Lisinopril",
      "adverse_reactions": [
        {
          "reaction": "dry persistent cough",
          "meddra_term": "Cough",
          "severity": "moderate",
          "known_adr": true
        },
        {
          "reaction": "dizziness when standing up quickly",
          "meddra_term": "Orthostatic hypotension",
          "severity": "mild",
          "known_adr": true
        }
      ],
      "urgency": "routine",
      "recommended_action": "Contact the prescriber soon to review likely ACE inhibitor cough and check blood pressure/orthostatic symptoms; seek urgent care if facial swelling, trouble breathing, fainting, or chest pain occur."
    },
    {
      "post_id": "2",
      "drug": "Methotrexate",
      "adverse_reactions": [
        {
          "reaction": "severe nausea after each dose",
          "meddra_term": "Nausea",
          "severity": "severe",
          "known_adr": true
        },
        {
          "reaction": "mouth sores",
          "meddra_term": "Stomatitis",
          "severity": "moderate",
          "known_adr": true
        },
        {
          "reaction": "liver enzymes 3x upper limit of normal",
          "meddra_term": "Hepatic enzyme increased",
          "severity": "severe",
          "known_adr": true
        },
        {
          "reaction": "fatigue",
          "meddra_term": "Fatigue",
          "severity": "moderate",
          "known_adr": true
        }
      ],
      "urgency": "urgent",
      "recommended_action": "Promptly contact the rheumatologist for reassessment; methotrexate toxicity is possible. Review need to hold the medication, repeat liver tests/CBC, and assess folate supplementation. Seek urgent evaluation sooner if fever, jaundice, bleeding, shortness of breath, or worsening weakness occurs."
    },
    {
      "post_id": "3",
      "drug": "Sertraline",
      "adverse_reactions": [
        {
          "reaction": "couldn't sleep",
          "meddra_term": "Insomnia",
          "severity": "moderate",
          "known_adr": true
        },
        {
          "reaction": "felt jittery",
          "meddra_term": "Feeling jittery",
          "severity": "mild",
          "known_adr": true
        },
        {
          "reaction": "weird vivid dreams",
          "meddra_term": "Abnormal dreams",
          "severity": "mild",
          "known_adr": true
        },
        {
          "reaction": "gained 12 pounds",
          "meddra_term": "Weight increased",
          "severity": "moderate",
          "known_adr": true
        },
        {
          "reaction": "libido is completely gone",
          "meddra_term": "Libido decreased",
          "severity": "moderate",
          "known_adr": true
        }
      ],
      "urgency": "routine",
      "recommended_action": "Discuss ongoing sexual dysfunction and weight gain with the prescriber at the next visit; dose adjustment or switching therapy may help. Seek prompt care if severe agitation, suicidal thoughts, or serotonin-syndrome symptoms develop."
    },
    {
      "post_id": "4",
      "drug": "Warfarin",
      "adverse_reactions": [
        {
          "reaction": "massive bruise on hip after fall",
          "meddra_term": "Contusion",
          "severity": "severe",
          "known_adr": true
        },
        {
          "reaction": "blood in urine for the past few days",
          "meddra_term": "Haematuria",
          "severity": "life-threatening",
          "known_adr": true
        }
      ],
      "urgency": "emergency",
      "recommended_action": "Immediate emergency evaluation is appropriate. Possible clinically significant bleeding on warfarin after trauma requires urgent assessment, INR testing, and management; hold further doses unless a clinician instructs otherwise."
    }
  ]
}
```

⏱ 28.5s | in: 393  out: 1084  total: 1477


## 9 · Multi-Turn Diagnostic Decision Support

Simulate a diagnostic assistant conversation using `previous_response_id` for multi-turn context.

In [10]:
SYSTEM = (
    "You are a diagnostic decision support assistant for physicians. "
    "Ask targeted clarifying questions, build a differential diagnosis, "
    "and suggest next steps. Be systematic and concise."
)

turns = [
    "45-year-old male, presents with 3-week history of progressive fatigue, unintentional "
    "weight loss of 8 lbs, and night sweats. No fever. Non-smoker, no significant PMH.",

    "CBC shows WBC 2.8 (low), Hgb 9.2 (low), platelets 95K (low). Peripheral smear shows "
    "circulating blasts at 12%. LDH elevated at 580. No lymphadenopathy on exam but "
    "splenomegaly noted.",

    "Flow cytometry confirms CD34+, CD33+, CD117+, MPO+. Cytogenetics pending. "
    "What is the most likely diagnosis and what should we do next?"
]

prev_id = None
for i, msg in enumerate(turns, 1):
    print(f"\n{'='*60}")
    print(f"TURN {i} — Physician input:")
    print(textwrap.fill(msg, width=80))
    print(f"{'='*60}")

    r, t = ask(msg, system=SYSTEM, prev_id=prev_id, effort="medium")
    prev_id = r.id
    show(r, t)


TURN 1 — Physician input:
45-year-old male, presents with 3-week history of progressive fatigue,
unintentional weight loss of 8 lbs, and night sweats. No fever. Non-smoker, no
significant PMH.


Key concern: **constitutional “B-type” symptoms** over 3 weeks — fatigue, weight loss, night sweats — which warrants evaluation for **hematologic malignancy, chronic infection, inflammatory disease, and endocrine/metabolic causes**.

## Targeted clarifying questions
Please clarify:

### 1) Symptom characterization
- How much weight loss relative to baseline body weight?
- Are the **night sweats drenching** (soaking clothes/sheets) or mild?
- Any **measured fevers**, chills, pruritus, or reduced appetite?
- Any **sleep disturbance**, depression, or stressors?

### 2) Localizing symptoms
- Any **lymph node enlargement** in neck/axilla/groin?
- Any **cough, dyspnea, chest pain, hemoptysis**?
- Any **abdominal pain/fullness**, early satiety, change in bowel habits, GI bleeding?
- Any **bone pain**, back pain, headaches, or focal neurologic symptoms?
- Any **recurrent infections**, easy bruising/bleeding?

### 3) Infection risk
- TB exposure, incarceration, homelessness, shelter work, healthcare exposure?
- Recent travel, animal exposures, tick bites?
- HIV/STI risk factors?
- Dental issues, endocarditis risk, IV drug use?
- Recent viral illness or mononucleosis-like symptoms?

### 4) Systemic/inflammatory/endocrine clues
- Heat intolerance, tremor, palpitations, diarrhea (hyperthyroidism)?
- Arthralgias, rash, oral ulcers, temporal headache, muscle weakness?
- Polyuria/polydipsia?

### 5) Exam findings / basic data if available
- Vitals?
- Any palpable nodes, hepatosplenomegaly, cachexia?
- CBC, CMP, ESR/CRP, LDH if already obtained?

---

## Preliminary differential diagnosis

### Higher-priority considerations
1. **Lymphoma (Hodgkin or non-Hodgkin)**
   - Weight loss + night sweats + fatigue are classic.
   - Ask about painless lymphadenopathy, pruritus, alcohol-induced node pain, cough/mediastinal symptoms.

2. **Leukemia or other hematologic malignancy**
   - Fatigue from anemia; sweats/weight loss possible.
   - Look for bruising, infections, bone pain, splenomegaly.

3. **Chronic infection**
   - **Tuberculosis** especially with night sweats/weight loss.
   - **HIV** with or without opportunistic infection.
   - Less commonly endocarditis, EBV/CMV, occult abscess depending on exposures.

4. **Solid malignancy**
   - Less specific, but unexplained weight loss/night sweats can be paraneoplastic or from advanced cancer.
   - Symptoms would help localize.

### Also consider
5. **Hyperthyroidism**
   - Weight loss, sweats, fatigue; usually palpitations/tremor/heat intolerance.

6. **Inflammatory/autoimmune disease**
   - E.g., vasculitis, connective tissue disease, sarcoid; usually other systemic symptoms.

7. **Less likely but possible**
   - Diabetes, adrenal disease, medication/substance-related causes, sleep disorders, mood disorder.

---

## Initial evaluation / next steps

### Focused physical exam
- Full lymph node exam
- Hepatosplenomegaly
- Skin exam (rash, bruising)
- Cardiac exam for murmur
- General cachexia and thyroid exam

### Initial labs
- **CBC with differential + peripheral smear**
- **CMP** (including liver enzymes, calcium, renal function)
- **ESR and/or CRP**
- **LDH**
- **TSH ± free T4**
- **HIV Ag/Ab**
- **TB testing** (IGRA preferred in many adults) if risk or no alternative explanation
- **Urinalysis**
- Consider **ferritin**, **B12/folate**, **retic count** if anemia found
- Blood cultures only if endocarditis/systemic infection suspected

### Initial imaging
- **Chest X-ray** as a low-threshold first step
- If abnormal exam/labs or persistent concern: **CT chest/abdomen/pelvis** to assess lymphadenopathy, occult malignancy, organomegaly

### If lymphadenopathy present
- Avoid assuming reactive etiology if persistent or with B symptoms.
- Consider **excisional lymph node biopsy** if suspicious nodes are found.

---

## Features that would increase urgency
- Abnormal CBC: blasts, cytopenias, marked leukocytosis
- Palpable generalized lymphadenopathy or splenomegaly
- Drenching night sweats, rapidly progressive weight loss
- Dyspnea/chest pressure suggesting mediastinal mass
- Hemoptysis, GI bleeding, severe weakness, fevers

If you want, I can help you **prioritize the differential after you provide exam findings/vitals** or build a **stepwise outpatient workup vs admission threshold**.

⏱ 32.5s | in: 83  out: 1177  total: 1260

TURN 2 — Physician input:
CBC shows WBC 2.8 (low), Hgb 9.2 (low), platelets 95K (low). Peripheral smear
shows circulating blasts at 12%. LDH elevated at 580. No lymphadenopathy on exam
but splenomegaly noted.


This is now **highly concerning for a hematologic malignancy, most likely acute leukemia or an advanced myeloid neoplasm**.

## Why this is urgent
- **Pancytopenia**: WBC 2.8, Hgb 9.2, platelets 95K
- **Circulating blasts (12%)**: abnormal and concerning for marrow malignancy
- **Elevated LDH**: high cell turnover
- **Splenomegaly + constitutional symptoms**

This patient needs **urgent hematology evaluation, likely hospital admission**, even without fever.

## Most likely differential
### 1) **Acute leukemia**
- **AML** is most likely in this age group
- **ALL** is less common but still possible
- Peripheral blasts + cytopenias + B symptoms strongly support this

### 2) **Myelodysplastic syndrome with excess blasts / evolving AML**
- Can present with cytopenias and circulating blasts
- Less dramatic than AML, but still urgent

### 3) **Other myeloid neoplasm in accelerated/blast phase**
- e.g. primary myelofibrosis or other MPN
- Splenomegaly supports this, though presentation is less classic

### 4) **Hairy cell leukemia or splenic lymphoma/leukemia**
- Splenomegaly + cytopenias fit
- But **circulating blasts** make these less likely than acute leukemia

### 5) **Less likely non-malignant marrow process**
- Severe viral marrow suppression, HLH, toxin/drug effect
- Much less likely if true blasts are confirmed

---

## Immediate questions that affect urgency
Please clarify:
- Any **fever**, rigors, or recent infections?
- Any **bleeding**, petechiae, epistaxis, gum bleeding, melena?
- Any **dyspnea, chest pain, severe weakness, syncope**
- Any **bone pain**
- Any prior CBCs showing chronic cytopenias?
- Any medication/toxin exposure: chemotherapy, benzene, chloramphenicol, linezolid, anticonvulsants?
- What is the **ANC** if available?

---

## Next steps: urgent workup
### 1) **Immediate hematology consult / admission**
Given circulating blasts and pancytopenia, this should generally be treated as **suspected acute leukemia until proven otherwise**.

### 2) Confirm and classify the process
- **Repeat CBC with differential**
- **Manual smear review by hematopathology**
- **Peripheral blood flow cytometry**
- **Bone marrow aspirate/biopsy** with:
  - morphology
  - flow cytometry
  - cytogenetics/karyotype
  - FISH / molecular testing

### 3) Labs needed now
- **CMP**
- **Uric acid, phosphorus, calcium, creatinine**  
  - assess tumor lysis risk
- **PT/INR, aPTT, fibrinogen, D-dimer**
  - especially important to evaluate for **DIC**, particularly if AML/APL is possible
- **Type and screen**
- Consider **haptoglobin, bilirubin, retic** if hemolysis question
- Viral testing often includes **HIV, hepatitis B/C**

### 4) If concern for acute promyelocytic leukemia (APL)
Need to know whether smear mentions **promyelocytes, Auer rods, heavy granulation**, or if coags are abnormal.  
If APL is suspected, this becomes a **hematologic emergency** because of life-threatening DIC.

### 5) Supportive management
- Neutropenic precautions if ANC is low
- Transfusion support as needed
- Avoid NSAIDs / unnecessary anticoagulation if thrombocytopenic
- If febrile: treat as **neutropenic fever** with prompt broad-spectrum IV antibiotics

---

## Bottom line
This presentation is **not consistent with a benign outpatient fatigue/weight-loss workup anymore**.  
The leading diagnosis is **acute leukemia (especially AML) vs MDS with excess blasts/evolving AML**, and the patient needs **same-day hematology-directed evaluation**.

If you want, I can help you with:
1. a **most likely leukemia subtype framework**, or  
2. a **specific admission order/workup checklist** for suspected acute leukemia.

⏱ 46.3s | in: 1217  out: 1430  total: 2647

TURN 3 — Physician input:
Flow cytometry confirms CD34+, CD33+, CD117+, MPO+. Cytogenetics pending. What
is the most likely diagnosis and what should we do next?


**Most likely diagnosis: Acute myeloid leukemia (AML).**

Why:
- **CD34+, CD117+**: immature myeloid blast phenotype
- **CD33+**: myeloid lineage
- **MPO+**: confirms **myeloid differentiation**
- Clinical picture fits: **fatigue, weight loss, night sweats, pancytopenia, circulating blasts, splenomegaly, elevated LDH**

## What to do next

### 1) Admit and involve hematology/oncology urgently
This is a **new AML** until proven otherwise.

### 2) Complete definitive diagnostic workup
If not already done:
- **Bone marrow aspirate/biopsy**
  - morphology/blast %
  - flow cytometry
  - conventional cytogenetics/karyotype
  - FISH
  - molecular panel

### 3) Send AML molecular studies now
These will guide prognosis and treatment:
- **FLT3-ITD/TKD**
- **NPM1**
- **CEBPA**
- **IDH1/IDH2**
- **TP53**
- **RUNX1, ASXL1**
- Broad myeloid NGS panel if available

### 4) Urgently exclude **APL**
Even though this phenotype is more typical of non-APL AML, **APL must be ruled out quickly** because of DIC risk.
- Check whether smear shows **promyelocytes/Auer rods**
- Send **PML-RARA testing** (FISH/PCR) if any suspicion
- Obtain **PT/INR, aPTT, fibrinogen, D-dimer**
- If morphology/coags strongly suggest APL, **start ATRA immediately while awaiting confirmation**

### 5) Supportive care now
- **Tumor lysis monitoring/prophylaxis**
  - CMP, uric acid, phosphorus, calcium, creatinine
  - IV fluids
  - **Allopurinol**; rasburicase if high uric acid/high TLS risk
- **Type and screen**
- Transfuse as needed:
  - PRBCs for symptomatic anemia
  - Platelets depending on count/bleeding/procedures
- If ANC is very low or fever develops:
  - treat as **neutropenic fever**
- Avoid unnecessary anticoagulation/NSAIDs if thrombocytopenic

### 6) Treatment planning
At **45 years old with no major PMH**, he is likely a candidate for **intensive induction therapy** if clinically fit.

Typical pathway:
- **“7+3” induction** (cytarabine + anthracycline) for fit patients
- Add targeted therapy depending on mutations:
  - e.g. **midostaurin** if **FLT3-mutated**
- If secondary/AML-MRC features emerge, regimen may differ (e.g. **CPX-351** in selected cases)
- If unfit, lower-intensity options include **azacitidine/decitabine + venetoclax**

### 7) Early transplant planning
Because he is relatively young:
- Start considering **allogeneic stem cell transplant candidacy**
- HLA typing / donor search may be initiated once risk stratification is clearer

---

## Bottom line
This immunophenotype is **most consistent with AML**.  
The next steps are:
1. **hospitalize / urgent heme-onc management**
2. **bone marrow + cytogenetic/molecular classification**
3. **rule out APL immediately**
4. **start supportive care and prepare for induction therapy**

If you want, I can next give you a **practical initial AML admission order set/checklist** or a **mutation-based treatment algorithm**.

⏱ 32.3s | in: 2168  out: 1091  total: 3259


---
# CROSS-DOMAIN

## 10 · Insurance Claim Medical Review Automation

Combine finance and healthcare: review an insurance claim with medical documentation for medical necessity and consistency.

In [11]:
claim = """
INSURANCE CLAIM #IC-2026-44891

CLAIMANT: Sarah K., 52F | Policy: PPO Gold | Claim Amount: $47,200
PROCEDURE: Lumbar spinal fusion (L4-L5), CPT 22630
PROVIDER: Dr. R. Patel, Orthopedic Spine Surgery, Metro General Hospital
DATE OF SERVICE: 2026-02-15

CLINICAL DOCUMENTATION:
- MRI (2025-11-01): L4-L5 disc herniation with moderate foraminal stenosis. No cord compression.
- Conservative treatment log: 8 weeks PT (2025-09 to 2025-11), 2 epidural steroid injections
  (2025-11-20 and 2025-12-18), NSAIDs x 4 months. Patient reports persistent radiculopathy
  with VAS pain score 7/10.
- Surgeon notes: \"Patient failed conservative management. Functional limitation in ADLs.
  Recommend single-level posterior lumbar interbody fusion.\"
- Pre-op clearance: Cardiology cleared (mild HTN, controlled). BMI 31.

BILLING DETAILS:
- Surgeon fee: $12,500 | Anesthesia: $4,200 | Facility fee: $28,000
- Implant costs: $8,500 (interbody cage + pedicle screws)
- Post-op follow-ups (3): $1,500 | PT referral: $2,500
- Total billed: $57,200 | Contracted rate: $47,200

RED FLAGS NOTED BY INITIAL REVIEW:
- Only 8 weeks of PT (guideline suggests 12 weeks minimum)
- No EMG/NCS to confirm radiculopathy
- Facility fee appears above 90th percentile for region
"""

claim_schema = {
    "name": "claim_review",
    "schema": {
        "type": "object",
        "properties": {
            "claim_id": {"type": "string"},
            "medical_necessity": {"type": "string"},
            "conservative_treatment_adequate": {"type": "boolean"},
            "documentation_gaps": {"type": "array", "items": {"type": "string"}},
            "billing_concerns": {"type": "array", "items": {"type": "string"}},
            "guideline_compliance": {"type": "array", "items": {
                "type": "object", "properties": {
                    "criterion": {"type": "string"},
                    "met": {"type": "boolean"},
                    "detail": {"type": "string"}
                }, "required": ["criterion","met","detail"], "additionalProperties": False
            }},
            "decision": {"type": "string"},
            "rationale": {"type": "string"},
            "additional_info_needed": {"type": "array", "items": {"type": "string"}}
        },
        "required": ["claim_id","medical_necessity","conservative_treatment_adequate",
                      "documentation_gaps","billing_concerns","guideline_compliance",
                      "decision","rationale","additional_info_needed"],
        "additionalProperties": False
    }
}

r, t = ask(
    f"Review this insurance claim for medical necessity, documentation completeness, "
    f"and billing appropriateness:\n\n{claim}",
    schema=claim_schema, effort="medium",
    system="You are a medical reviewer for a health insurance company. Apply evidence-based guidelines for lumbar fusion. Be thorough but fair."
)
show(r, t, raw_json=True)

```json
{
  "claim_id": "IC-2026-44891",
  "medical_necessity": "Lumbar fusion at L4-L5 is not sufficiently supported as medically necessary based on the submitted record. The documentation shows an L4-L5 disc herniation with moderate foraminal stenosis and persistent radicular pain despite nonoperative care, which may support surgical decompression/discectomy in selected cases. However, fusion generally requires additional justification such as segmental instability, spondylolisthesis, deformity, recurrent disc herniation after prior surgery, symptomatic pseudarthrosis, or other clearly documented indications. None of these indications are documented here.",
  "conservative_treatment_adequate": false,
  "documentation_gaps": [
    "No documentation of lumbar instability (e.g., flexion-extension radiographs) or spondylolisthesis to justify fusion rather than decompression alone.",
    "No prior lumbar surgery or recurrent disc herniation documented.",
    "No objective neurologic exam findings provided (motor deficit, reflex changes, dermatomal sensory loss, positive tension signs).",
    "Functional impairment is stated generally ('limitation in ADLs') but no standardized disability measure is included (e.g., ODI, PROMIS, Roland-Morris).",
    "Conservative treatment is partially documented, but supervised PT duration is only 8 weeks; many fusion guidelines expect at least 12 weeks and/or a longer documented course unless urgent neurologic compromise exists.",
    "No detailed rationale from the surgeon explaining why posterior lumbar interbody fusion was chosen over decompression/discectomy alone.",
    "No evidence of urgent indications such as cauda equina syndrome, progressive motor deficit, fracture, tumor, infection, or severe instability.",
    "No smoking status documented, which is relevant to fusion risk stratification and perioperative planning.",
    "No EMG/NCS provided; while not always required when imaging and exam correlate, its absence leaves the radiculopathy confirmation less complete given limited neurologic exam documentation."
  ],
  "billing_concerns": [
    "Facility fee of $28,000 is flagged as above the 90th percentile for the region and should be validated against contracted hospital reimbursement terms and implant carve-out rules.",
    "Post-operative follow-up visits may fall within the global surgical package and may not be separately reimbursable unless documentation supports separately billable services.",
    "'PT referral' billed at $2,500 is unclear and may not be an appropriate separately reimbursable charge; actual rendered PT services, if any, should be itemized and billed by the treating therapy provider under applicable codes.",
    "Implant costs ($8,500) should be reviewed to confirm whether they are separately payable or already included in the facility reimbursement under the contract.",
    "Claim amount and billing presentation should be reconciled: total billed charges are $57,200, while the claim amount requested is the contracted rate of $47,200."
  ],
  "guideline_compliance": [
    {
      "criterion": "Imaging demonstrates pathology at the proposed operative level",
      "met": true,
      "detail": "MRI shows L4-L5 disc herniation with moderate foraminal stenosis, anatomically consistent with possible radicular symptoms."
    },
    {
      "criterion": "Persistent symptoms despite conservative management",
      "met": true,
      "detail": "NSAIDs for 4 months, 8 weeks of PT, and 2 epidural steroid injections are documented, with ongoing reported radicular pain VAS 7/10."
    },
    {
      "criterion": "Adequate duration/intensity of conservative treatment for lumbar fusion",
      "met": false,
      "detail": "Only 8 weeks of PT are documented; many evidence-based fusion policies require at least 12 weeks and/or a more robust trial before fusion unless exceptions apply."
    },
    {
      "criterion": "Objective neurologic findings correlating with imaging",
      "met": false,
      "detail": "The file does not include a detailed neurologic examination documenting motor, sensory, reflex, or tension-sign abnormalities."
    },
    {
      "criterion": "Accepted indication for lumbar fusion beyond simple disc herniation/stenosis",
      "met": false,
      "detail": "No documented instability, spondylolisthesis, deformity, recurrent herniation after prior surgery, traumatic/postinfectious condition, or other standard indication for interbody fusion."
    },
    {
      "criterion": "Documentation explaining why fusion is preferred over decompression alone",
      "met": false,
      "detail": "Surgeon note states conservative treatment failure but does not justify the need for fusion specifically."
    },
    {
      "criterion": "Urgent or exception criteria allowing shorter conservative care",
      "met": false,
      "detail": "No progressive neurologic deficit, cauda equina syndrome, infection, fracture, tumor, or other urgent indication is documented."
    },
    {
      "criterion": "EMG/NCS confirmation of radiculopathy",
      "met": false,
      "detail": "Not provided. This is not universally mandatory if clinical and imaging correlation are strong, but current documentation does not otherwise establish that correlation robustly."
    }
  ],
  "decision": "Deny as not medically necessary as submitted; reconsider on appeal with additional supporting documentation.",
  "rationale": "The submitted record supports lumbar radiculopathy from L4-L5 pathology and shows a meaningful trial of conservative care. However, for CPT 22630 lumbar interbody fusion, the documentation does not establish a generally accepted indication for fusion. Disc herniation with moderate foraminal stenosis alone, without documented instability, spondylolisthesis, deformity, recurrent herniation, prior failed decompression, or another fusion-specific indication, is typically insufficient to justify fusion. The chart also lacks a detailed neurologic exam and a clear explanation of why decompression/discectomy alone would be inadequate. In addition, conservative management documentation is somewhat short of common fusion-policy expectations due to only 8 weeks of PT. Billing also requires audit attention for the high facility charge, unclear PT referral charge, implant reimbursement status, and possible global-package issues for post-op follow-ups.",
  "additional_info_needed": [
    "Comprehensive preoperative neurologic examination documenting objective deficits and correlation to the L4-L5 level.",
    "Surgeon narrative specifically explaining why fusion was required instead of decompression/discectomy alone.",
    "Flexion-extension lumbar radiographs or other evidence demonstrating instability, if present.",
    "Any documentation of spondylolisthesis, deformity, recurrent herniation, severe disc-space collapse/instability, or prior failed decompression.",
    "Standardized functional outcome measure (e.g., ODI) showing degree of disability before surgery.",
    "Complete conservative treatment records, including PT notes, home exercise compliance, medication response, and duration rationale if PT was stopped before 12 weeks.",
    "Smoking/nicotine status and perioperative optimization documentation.",
    "Itemized billing support for facility charges, implant carve-out terms, and clarification of the $2,500 PT referral charge.",
    "Support for any separately billed post-op follow-up services outside the global surgical package."
  ]
}
```

⏱ 62.6s | in: 566  out: 1865  total: 2431
